# Experiment: SpectralBio Block 12D - Final Nuclear Localization (H100)

Objective:
- Treat Block 12D as the **last notebook** and throw every legitimate adjudication tool at the remaining gap.
- Start from the evidence already assembled in **12B** and **12C**, then search for a **localized covariance-native regime** that is strong enough to become nuclear even if the global claim stays bounded.
- Force the notebook to answer the question the paper now hinges on: **is there at least one compact, coverage-aware, scalar-beating, structurally witnessed rescue regime that deserves primary status?**
- Produce a final package under `New Notebooks/results/12d_block12_final_nuclear_localization_h100/` with tables, figures, manifests, and a bundle ready for reviewer-facing use.


## Final Contract

- Block 12C showed that the story is stronger than a single checkpoint but still not nuclear at the global level.
- Block 12D therefore changes the target from **global universality** to **localized nuclearization** under a stricter but more honest contract.
- A model may now win the primary claim only if a **localized panel** survives all of the following together:
  1. a coverage floor computed on the localized panel,
  2. positive covariance enrichment,
  3. wins over global scalar matched controls,
  4. wins over local scalar matched controls,
  5. wins over chemistry-only controls,
  6. same-position evidence that is positive, tied, or genuinely not applicable,
  7. a structural witness from the strict TP53 layer.
- The notebook is allowed to conclude that the best final claim is **localized and bounded**, but it is not allowed to hide behind weak controls or empty transfer stories.


In [1]:
# Setup: imports, runtime requirements, notebook identifiers, and the adjudication roster
from __future__ import annotations

import importlib
import json
import math
import os
import platform
import random
import subprocess
import shutil
import sys
import time
import zipfile
from importlib import metadata as importlib_metadata
from datetime import datetime, timezone
from pathlib import Path

from IPython.display import display

NOTEBOOK_SLUG = '12d_block12_final_nuclear_localization_h100'
BLOCK12B_SLUG = '12b_block12_multifamily_coverage_aware_generalization_h100'
ACCOUNT_LABEL = os.environ.get('SPECTRALBIO_ACCOUNT_LABEL', 'local_run')
RUN_AT = datetime.now(timezone.utc).isoformat()
OVERWRITE = os.environ.get('SPECTRALBIO_OVERWRITE', '').strip().lower() in {'1', 'true', 'yes'}
RERUN_BRCA1 = os.environ.get('SPECTRALBIO_BLOCK12D_RERUN_BRCA1', '1').strip().lower() in {'1', 'true', 'yes'}
INCLUDE_ANKH = os.environ.get('SPECTRALBIO_BLOCK12D_INCLUDE_ANKH', '').strip().lower() in {'1', 'true', 'yes'}
WINDOW_RADIUS = int(os.environ.get('SPECTRALBIO_BLOCK12D_WINDOW_RADIUS', '40'))
CHECKPOINT_EVERY = int(os.environ.get('SPECTRALBIO_BLOCK12D_CHECKPOINT_EVERY', '10'))
FIXED_ALPHA = float(os.environ.get('SPECTRALBIO_BLOCK12D_FIXED_ALPHA', '0.55'))
BOOTSTRAP_REPLICATES = int(os.environ.get('SPECTRALBIO_BLOCK12D_BOOTSTRAP_REPLICATES', '2000'))
RANDOM_REPLICATES = int(os.environ.get('SPECTRALBIO_BLOCK12D_RANDOM_REPLICATES', '500'))
RANDOM_SEED = int(os.environ.get('SPECTRALBIO_BLOCK12D_RANDOM_SEED', '42'))
MIN_RULE_ON_ABS = int(os.environ.get('SPECTRALBIO_BLOCK12D_MIN_RULE_ON_ABS', '10'))
MIN_RULE_ON_FRAC = float(os.environ.get('SPECTRALBIO_BLOCK12D_MIN_RULE_ON_FRAC', '0.05'))
MODEL_FILTER = {
    item.strip().lower()
    for item in os.environ.get('SPECTRALBIO_BLOCK12D_MODEL_FILTER', '').split(',')
    if item.strip()
}

MODEL_SPECS = [
    {
        'model_name': 'facebook/esm2_t12_35M_UR50D',
        'model_label': 'ESM2-35M',
        'family_label': 'esm',
        'architecture_kind': 'masked_encoder',
        'adapter_kind': 'esm',
        'sequence_mode': 'raw',
        'scale_bucket': '35M',
    },
    {
        'model_name': 'facebook/esm2_t30_150M_UR50D',
        'model_label': 'ESM2-150M',
        'family_label': 'esm',
        'architecture_kind': 'masked_encoder',
        'adapter_kind': 'esm',
        'sequence_mode': 'raw',
        'scale_bucket': '150M',
    },
    {
        'model_name': 'facebook/esm2_t33_650M_UR50D',
        'model_label': 'ESM2-650M',
        'family_label': 'esm',
        'architecture_kind': 'masked_encoder',
        'adapter_kind': 'esm',
        'sequence_mode': 'raw',
        'scale_bucket': '650M',
    },
    {
        'model_name': 'facebook/esm2_t36_3B_UR50D',
        'model_label': 'ESM2-3B',
        'family_label': 'esm',
        'architecture_kind': 'masked_encoder',
        'adapter_kind': 'esm',
        'sequence_mode': 'raw',
        'scale_bucket': '3B',
    },
    {
        'model_name': 'facebook/esm1v_t33_650M_UR90S_1',
        'model_label': 'ESM-1v',
        'family_label': 'esm_variant_specialist',
        'architecture_kind': 'masked_encoder',
        'adapter_kind': 'esm',
        'sequence_mode': 'raw',
        'scale_bucket': '650M',
    },
    {
        'model_name': 'Rostlab/prot_t5_xl_half_uniref50-enc',
        'model_label': 'ProtT5',
        'family_label': 'prottrans',
        'architecture_kind': 'encoder_decoder',
        'adapter_kind': 't5',
        'sequence_mode': 'spaced',
        'scale_bucket': 'xl',
    },
    {
        'model_name': 'Rostlab/prot_bert',
        'model_label': 'ProtBERT',
        'family_label': 'bert_style',
        'architecture_kind': 'bidirectional_encoder',
        'adapter_kind': 'auto',
        'sequence_mode': 'spaced',
        'scale_bucket': 'base',
    },
    {
        'model_name': 'Rostlab/prot_bert_bfd',
        'model_label': 'ProtBERT-BFD',
        'family_label': 'bert_style',
        'architecture_kind': 'bidirectional_encoder',
        'adapter_kind': 'auto',
        'sequence_mode': 'spaced',
        'scale_bucket': 'bfd',
    },
    {
        'model_name': 'hugohrban/progen2-small',
        'model_label': 'ProGen2-small',
        'family_label': 'progen',
        'architecture_kind': 'causal_decoder',
        'adapter_kind': 'causal',
        'sequence_mode': 'raw',
        'scale_bucket': 'small',
    },
    {
        'model_name': 'hugohrban/progen2-base',
        'model_label': 'ProGen2-base',
        'family_label': 'progen',
        'architecture_kind': 'causal_decoder',
        'adapter_kind': 'causal',
        'sequence_mode': 'raw',
        'scale_bucket': 'base',
    },
]

if INCLUDE_ANKH:
    MODEL_SPECS.extend([
        {
            'model_name': 'ElnaggarLab/ankh-base',
            'model_label': 'Ankh-base',
            'family_label': 'ankh',
            'architecture_kind': 'encoder_decoder',
            'adapter_kind': 'auto',
            'sequence_mode': 'spaced',
            'scale_bucket': 'base',
        },
        {
            'model_name': 'ElnaggarLab/ankh-large',
            'model_label': 'Ankh-large',
            'family_label': 'ankh',
            'architecture_kind': 'encoder_decoder',
            'adapter_kind': 'auto',
            'sequence_mode': 'spaced',
            'scale_bucket': 'large',
        },
    ])

if MODEL_FILTER:
    MODEL_SPECS = [
        spec
        for spec in MODEL_SPECS
        if spec['model_label'].strip().lower() in MODEL_FILTER
        or spec['family_label'].strip().lower() in MODEL_FILTER
    ]

def done(message: str) -> None:
    print(message)
    print('TERMINEI PODE SEGUIR')

try:
    from packaging.requirements import Requirement as _Requirement
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'packaging>=24.0'], check=True, text=True)
    from packaging.requirements import Requirement as _Requirement

TORCH_SPEC = 'torch>=2.5,<3'
runtime_requirements = [
    ('packaging', 'packaging>=24.0', 'packaging'),
    ('numpy', 'numpy==2.1.3', 'numpy'),
    ('pandas', 'pandas>=2.2.0', 'pandas'),
    ('matplotlib', 'matplotlib>=3.9.0', 'matplotlib'),
    ('sklearn', 'scikit-learn==1.5.2', 'sklearn'),
    ('scipy', 'scipy==1.14.1', 'scipy'),
    ('requests', 'requests>=2.32.0', 'requests'),
    ('torch', TORCH_SPEC, 'torch'),
    ('transformers', 'transformers==4.49.0', 'transformers'),
    ('accelerate', 'accelerate>=1.0.0', 'accelerate'),
    ('sentencepiece', 'sentencepiece>=0.2.0', 'sentencepiece'),
    ('safetensors', 'safetensors>=0.4.0', 'safetensors'),
    ('protobuf', 'protobuf>=5.0.0', 'google.protobuf'),
    ('einops', 'einops>=0.8.0', 'einops'),
]

def parse_requirement(spec: str):
    requirement = _Requirement(spec)
    return requirement.name, requirement.specifier

def installed_version_for(requirement_spec: str) -> str | None:
    package_name, _ = parse_requirement(requirement_spec)
    try:
        return importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        return None

def requirement_satisfied(requirement_spec: str) -> bool:
    version = installed_version_for(requirement_spec)
    if version is None:
        return False
    _, specifier = parse_requirement(requirement_spec)
    return True if not specifier else version in specifier

install_specs: list[str] = []
runtime_rows = []
for module_name, package_spec, import_target in runtime_requirements:
    satisfied = requirement_satisfied(package_spec)
    runtime_rows.append({
        'module': module_name,
        'package_spec': package_spec,
        'status': 'present' if satisfied else 'install_required',
        'version': installed_version_for(package_spec) or 'missing',
    })
    if not satisfied:
        install_specs.append(package_spec)

if install_specs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', *install_specs], check=True, text=True)
    importlib.invalidate_caches()
    runtime_rows = []
    for module_name, package_spec, import_target in runtime_requirements:
        module = importlib.import_module(import_target)
        runtime_rows.append({
            'module': module_name,
            'package_spec': package_spec,
            'status': 'installed_now',
            'version': str(getattr(module, '__version__', 'present')),
        })

print({
    'notebook_slug': NOTEBOOK_SLUG,
    'block12b_slug': BLOCK12B_SLUG,
    'account_label': ACCOUNT_LABEL,
    'rerun_brca1': RERUN_BRCA1,
    'include_ankh': INCLUDE_ANKH,
    'window_radius': WINDOW_RADIUS,
    'checkpoint_every': CHECKPOINT_EVERY,
    'fixed_alpha': FIXED_ALPHA,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'runtime': runtime_rows,
    'models': [spec['model_label'] for spec in MODEL_SPECS],
})
done('Environment prepared for Block 12D adjudication and structural closure.')

print("TERMINEI PODE SEGUIR")


{'notebook_slug': '12d_block12_final_nuclear_localization_h100', 'block12b_slug': '12b_block12_multifamily_coverage_aware_generalization_h100', 'account_label': 'local_run', 'rerun_brca1': False, 'include_ankh': False, 'window_radius': 40, 'checkpoint_every': 10, 'fixed_alpha': 0.55, 'python': '3.13.5', 'platform': 'Windows-11-10.0.26200-SP0', 'runtime': [{'module': 'packaging', 'package_spec': 'packaging>=24.0', 'status': 'present', 'version': '25.0'}, {'module': 'numpy', 'package_spec': 'numpy==2.1.3', 'status': 'present', 'version': '2.1.3'}, {'module': 'pandas', 'package_spec': 'pandas>=2.2.0', 'status': 'present', 'version': '2.3.3'}, {'module': 'matplotlib', 'package_spec': 'matplotlib>=3.9.0', 'status': 'present', 'version': '3.10.8'}, {'module': 'sklearn', 'package_spec': 'scikit-learn==1.5.2', 'status': 'present', 'version': '1.5.2'}, {'module': 'scipy', 'package_spec': 'scipy==1.14.1', 'status': 'present', 'version': '1.14.1'}, {'module': 'requests', 'package_spec': 'requests

In [2]:
# Helpers: repo discovery, statistics, structural closure, sequence validation, and optional live scoring
import json
import math
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy import stats
from sklearn.metrics import roc_auc_score
from transformers import AutoModel, AutoModelForMaskedLM, AutoModelForSeq2SeqLM, AutoTokenizer

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'New Notebooks').exists()

def find_repo_root() -> Path:
    env_repo = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()
    candidates: list[Path] = []
    if env_repo:
        candidates.append(Path(env_repo).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/teamspace/studios/this_studio/Stanford-Claw4s'),
        Path('/teamspace/studios/this_studio/SpectralBio'),
        Path('/content/Stanford-Claw4s'),
    ])
    for candidate in candidates:
        if looks_like_repo(candidate):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate repo root. Set SPECTRALBIO_REPO_ROOT.')

REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from spectralbio.supplementary.reject_recovery import _ensure_gene_score_rows

RESULTS_DIR = REPO_ROOT / 'New Notebooks' / 'results'
RESULTS_ROOT = ensure_dir(RESULTS_DIR / NOTEBOOK_SLUG)
TABLES_DIR = ensure_dir(RESULTS_ROOT / 'tables')
FIGURES_DIR = ensure_dir(RESULTS_ROOT / 'figures')
TEXT_DIR = ensure_dir(RESULTS_ROOT / 'text')
MANIFESTS_DIR = ensure_dir(RESULTS_ROOT / 'manifests')
RUNTIME_DIR = ensure_dir(RESULTS_ROOT / 'runtime')
LIVE_SCORES_DIR = ensure_dir(RESULTS_ROOT / 'live_scores')
BUNDLE_ROOT = RESULTS_DIR / f'{NOTEBOOK_SLUG}_bundle'
ZIP_PATH = RESULTS_ROOT / f'{NOTEBOOK_SLUG}_bundle.zip'

BLOCK12B_ROOT = RESULTS_DIR / BLOCK12B_SLUG
BLOCK12B_TABLES_DIR = BLOCK12B_ROOT / 'tables'
BLOCK12B_MANIFESTS_DIR = BLOCK12B_ROOT / 'manifests'

def run(cmd: list[str], cwd: Path | None = None) -> str:
    completed = subprocess.run(cmd, cwd=str(cwd) if cwd else None, capture_output=True, text=True, check=True)
    return completed.stdout.strip()

def safe_float(value, default=float('nan')) -> float:
    try:
        numeric = float(value)
        if math.isnan(numeric):
            return default
        return numeric
    except Exception:
        return default

def weighted_mean(values, weights) -> float:
    values_arr = np.asarray(values, dtype=float)
    weights_arr = np.asarray(weights, dtype=float)
    valid = np.isfinite(values_arr) & np.isfinite(weights_arr) & (weights_arr > 0)
    if not valid.any():
        return float('nan')
    return float(np.average(values_arr[valid], weights=weights_arr[valid]))

def coverage_floor(n_variants: int) -> int:
    return max(MIN_RULE_ON_ABS, int(math.ceil(float(n_variants) * MIN_RULE_ON_FRAC)))

def bootstrap_gap(labels, mask, replicates: int, seed: int) -> dict:
    labels_arr = np.asarray(labels, dtype=float)
    mask_arr = np.asarray(mask, dtype=bool)
    if labels_arr.size == 0:
        return {
            'pathogenic_fraction_rule_on': float('nan'),
            'pathogenic_fraction_rule_off': float('nan'),
            'enrichment_gap': float('nan'),
            'gap_ci_low': float('nan'),
            'gap_ci_high': float('nan'),
        }
    on = labels_arr[mask_arr]
    off = labels_arr[~mask_arr]
    on_mean = float(np.mean(on)) if on.size else float('nan')
    off_mean = float(np.mean(off)) if off.size else float('nan')
    gap = on_mean - off_mean if on.size and off.size else float('nan')
    if not on.size or not off.size:
        return {
            'pathogenic_fraction_rule_on': on_mean,
            'pathogenic_fraction_rule_off': off_mean,
            'enrichment_gap': gap,
            'gap_ci_low': float('nan'),
            'gap_ci_high': float('nan'),
        }
    rng = np.random.default_rng(seed)
    samples = []
    for _ in range(replicates):
        sample_on = rng.choice(on, size=on.size, replace=True)
        sample_off = rng.choice(off, size=off.size, replace=True)
        samples.append(float(sample_on.mean() - sample_off.mean()))
    return {
        'pathogenic_fraction_rule_on': on_mean,
        'pathogenic_fraction_rule_off': off_mean,
        'enrichment_gap': gap,
        'gap_ci_low': float(np.quantile(samples, 0.025)),
        'gap_ci_high': float(np.quantile(samples, 0.975)),
    }

def odds_ratio_from_mask(labels, mask) -> dict:
    labels_arr = np.asarray(labels, dtype=int)
    mask_arr = np.asarray(mask, dtype=bool)
    positive_with = int(((labels_arr == 1) & mask_arr).sum())
    negative_with = int(((labels_arr == 0) & mask_arr).sum())
    positive_without = int(((labels_arr == 1) & ~mask_arr).sum())
    negative_without = int(((labels_arr == 0) & ~mask_arr).sum())
    odds_ratio = ((positive_with + 0.5) * (negative_without + 0.5)) / ((negative_with + 0.5) * (positive_without + 0.5))
    return {
        'positive_with_rule': positive_with,
        'negative_with_rule': negative_with,
        'positive_without_rule': positive_without,
        'negative_without_rule': negative_without,
        'odds_ratio': float(odds_ratio),
    }

def random_coverage_matched_gap(labels, n_on: int, replicates: int, seed: int) -> dict:
    labels_arr = np.asarray(labels, dtype=float)
    n_total = labels_arr.size
    if n_total == 0 or n_on <= 0 or n_on >= n_total:
        return {
            'random_gap_mean': float('nan'),
            'random_gap_p95': float('nan'),
            'random_gap_p05': float('nan'),
        }
    rng = np.random.default_rng(seed)
    gaps = []
    all_indices = np.arange(n_total)
    for _ in range(replicates):
        chosen = rng.choice(all_indices, size=n_on, replace=False)
        mask = np.zeros(n_total, dtype=bool)
        mask[chosen] = True
        on = labels_arr[mask]
        off = labels_arr[~mask]
        gaps.append(float(on.mean() - off.mean()))
    return {
        'random_gap_mean': float(np.mean(gaps)),
        'random_gap_p95': float(np.quantile(gaps, 0.95)),
        'random_gap_p05': float(np.quantile(gaps, 0.05)),
    }

def normalize_protein_sequence(sequence: str, sequence_mode: str) -> str:
    sequence = ''.join(str(sequence).strip().split()).upper()
    return ' '.join(sequence) if sequence_mode == 'spaced' else sequence

AA_BUCKETS = {
    'A': 'hydrophobic', 'V': 'hydrophobic', 'I': 'hydrophobic', 'L': 'hydrophobic', 'M': 'hydrophobic',
    'F': 'aromatic', 'Y': 'aromatic', 'W': 'aromatic',
    'S': 'polar', 'T': 'polar', 'N': 'polar', 'Q': 'polar', 'C': 'polar',
    'K': 'positive', 'R': 'positive', 'H': 'positive',
    'D': 'negative', 'E': 'negative',
    'G': 'special', 'P': 'special',
}

def aa_bucket(aa: str) -> str:
    return AA_BUCKETS.get(str(aa).upper(), 'other')

def chemistry_trigger_from_columns(frame: pd.DataFrame) -> pd.Series:
    wt_bucket = frame['wt_aa'].astype(str).str.upper().map(aa_bucket)
    mut_bucket = frame['mut_aa'].astype(str).str.upper().map(aa_bucket)
    return wt_bucket.ne(mut_bucket) | frame['mut_aa'].astype(str).str.upper().isin({'P', 'G', 'C'})

def select_hidden_layers(hidden_states) -> list:
    n_layers = len(hidden_states)
    if n_layers <= 4:
        return list(hidden_states)
    return [hidden_states[0], hidden_states[n_layers // 2], hidden_states[-1]]

def covariance_features_dual(wt_hidden: np.ndarray, mut_hidden: np.ndarray) -> dict:
    wt_last = np.asarray(wt_hidden[-1], dtype=np.float32)
    mut_last = np.asarray(mut_hidden[-1], dtype=np.float32)
    wt_centered = wt_last - wt_last.mean(axis=0, keepdims=True)
    mut_centered = mut_last - mut_last.mean(axis=0, keepdims=True)
    denom_wt = max(1, wt_centered.shape[0] - 1)
    denom_mut = max(1, mut_centered.shape[0] - 1)
    cov_wt = (wt_centered.T @ wt_centered) / denom_wt
    cov_mut = (mut_centered.T @ mut_centered) / denom_mut
    diff = cov_mut - cov_wt
    frob_dist = float(np.linalg.norm(diff, ord='fro'))
    trace_wt = float(np.trace(cov_wt))
    trace_mut = float(np.trace(cov_mut))
    trace_ratio = trace_mut / trace_wt if abs(trace_wt) > 1e-8 else float('nan')
    sign, logdet_wt = np.linalg.slogdet(cov_wt + np.eye(cov_wt.shape[0], dtype=np.float32) * 1e-4)
    sign_mut, logdet_mut = np.linalg.slogdet(cov_mut + np.eye(cov_mut.shape[0], dtype=np.float32) * 1e-4)
    sps_log = float(logdet_mut - logdet_wt) if sign > 0 and sign_mut > 0 else float('nan')
    return {
        'frob_dist': frob_dist,
        'trace_ratio': trace_ratio,
        'sps_log': sps_log,
    }

def score_vector_from_frame(frame: pd.DataFrame, alpha: float) -> pd.DataFrame:
    scored = frame.copy()
    ll_norm = scored['ll_proper'].rank(method='average', ascending=False, pct=True).astype(float)
    frob_norm = scored['frob_dist'].rank(method='average', ascending=False, pct=True).astype(float)
    scored['ll_rank_norm'] = ll_norm
    scored['frob_rank_norm'] = frob_norm
    scored['pair_rank_fixed_055'] = alpha * frob_norm + (1.0 - alpha) * ll_norm
    scored['pair_minus_ll'] = scored['pair_rank_fixed_055'] - scored['ll_rank_norm']
    scored['chemistry_trigger'] = chemistry_trigger_from_columns(scored)
    return scored

def compute_rule_mask(frame: pd.DataFrame, rule_type: str, threshold: float, confidence_threshold: float | None) -> pd.Series:
    pair_signal = pd.to_numeric(frame['pair_minus_ll'], errors='coerce').fillna(-999.0) >= float(threshold)
    scalar_signal = pd.to_numeric(frame['ll_rank_norm'], errors='coerce').fillna(-999.0) >= (1.0 - float(threshold))
    chemistry_signal = frame['chemistry_trigger'].fillna(False).astype(bool)
    confidence_signal = pd.to_numeric(frame['pair_rank_fixed_055'], errors='coerce').fillna(-999.0) >= float(confidence_threshold if confidence_threshold is not None else 0.0)

    if rule_type == 'chemistry_only':
        return chemistry_signal
    if rule_type == 'pair_only':
        return pair_signal
    if rule_type == 'scalar_only':
        return scalar_signal
    if rule_type == 'combined':
        return chemistry_signal & pair_signal
    if rule_type == 'combined_confident':
        return chemistry_signal & pair_signal & confidence_signal
    return pd.Series(np.zeros(len(frame), dtype=bool), index=frame.index)

def summarize_rule(frame: pd.DataFrame, rule_type: str, threshold: float, confidence_threshold: float | None, seed: int) -> dict:
    mask = compute_rule_mask(frame, rule_type, threshold, confidence_threshold)
    labels = frame['label'].astype(int).to_numpy()
    coverage_target = coverage_floor(len(frame))
    gap_stats = bootstrap_gap(labels, mask.to_numpy(), BOOTSTRAP_REPLICATES, seed)
    or_stats = odds_ratio_from_mask(labels, mask.to_numpy())
    random_stats = random_coverage_matched_gap(labels, int(mask.sum()), RANDOM_REPLICATES, seed + 17)
    return {
        'rule_type': str(rule_type),
        'threshold': float(threshold),
        'confidence_threshold': float(confidence_threshold) if confidence_threshold is not None else float('nan'),
        'n_rule_on': int(mask.sum()),
        'fraction_rule_on': float(mask.mean()),
        'coverage_target': int(coverage_target),
        'coverage_floor_pass': bool(int(mask.sum()) >= int(coverage_target)),
        **gap_stats,
        **or_stats,
        **random_stats,
    }

def select_covariance_candidate(candidate_df: pd.DataFrame) -> pd.Series | None:
    if candidate_df.empty:
        return None
    covariance_only = candidate_df.loc[candidate_df['rule_type'].isin(['pair_only', 'combined', 'combined_confident'])].copy()
    if covariance_only.empty:
        return None
    covariance_only['coverage_pass_int'] = covariance_only['coverage_floor_pass'].fillna(False).astype(int)
    covariance_only['rule_priority'] = covariance_only['rule_type'].map({
        'combined_confident': 0,
        'combined': 1,
        'pair_only': 2,
    }).fillna(99).astype(int)
    covariance_only = covariance_only.sort_values(
        ['coverage_pass_int', 'enrichment_gap', 'odds_ratio', 'n_rule_on', 'rule_priority'],
        ascending=[False, False, False, False, True],
    ).reset_index(drop=True)
    return covariance_only.iloc[0]

def same_position_top_hit_rate(frame: pd.DataFrame, score_column: str) -> dict:
    working = frame.copy()
    working[score_column] = pd.to_numeric(working[score_column], errors='coerce')
    grouped = []
    for position, group in working.groupby('position', sort=True):
        labels = group['label'].astype(int)
        if labels.nunique() < 2:
            continue
        top_row = group.sort_values(score_column, ascending=False, kind='stable').iloc[0]
        grouped.append(int(top_row['label']))
    if not grouped:
        return {'n_mixed_positions': 0, 'top_hit_pathogenic_rate': float('nan')}
    return {
        'n_mixed_positions': int(len(grouped)),
        'top_hit_pathogenic_rate': float(np.mean(grouped)),
    }

def safe_auc(labels, scores) -> float:
    labels_arr = pd.Series(labels).astype(int)
    scores_arr = pd.Series(scores).astype(float)
    if labels_arr.nunique() < 2:
        return float('nan')
    return float(roc_auc_score(labels_arr.to_numpy(), scores_arr.to_numpy()))

def tp53_domain_label(position: int) -> str:
    if 94 <= position <= 292:
        return 'dna_binding'
    if 323 <= position <= 355:
        return 'tetramerization'
    if 63 <= position <= 92:
        return 'proline_rich'
    return 'other'

TP53_HOTSPOTS = {175, 179, 220, 245, 248, 249, 273, 282}

def structural_reference_candidates() -> list[Path]:
    return [
        RESULTS_DIR / '07b_block10_structural_dissociation_tp53_h100' / '07b_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_strict.csv',
        RESULTS_DIR / '07b_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_strict.csv',
        RESULTS_DIR / '07c_block10_structural_dissociation_tp53_h100' / '07c_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_strict.csv',
        RESULTS_DIR / '07c_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_strict.csv',
        RESULTS_DIR / '07b_block10_structural_dissociation_tp53_h100' / '07b_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_broad.csv',
        RESULTS_DIR / '07b_block10_structural_dissociation_tp53_h100' / 'tables' / 'tp53_structural_pairs_variant_level_broad.csv',
    ]

def load_structural_reference() -> tuple[pd.DataFrame, str, str]:
    for candidate in structural_reference_candidates():
        if candidate.exists():
            frame = pd.read_csv(candidate)
            if 'contact_change_fraction_median' not in frame.columns and 'contact_change_fraction' in frame.columns:
                frame['contact_change_fraction_median'] = frame['contact_change_fraction']
            if {'wt_aa', 'mut_aa'}.issubset(frame.columns) and 'position_1' in frame.columns:
                frame['variant_id_one_based'] = (
                    'TP53:'
                    + frame['wt_aa'].astype(str).str.upper()
                    + frame['position_1'].astype(int).astype(str)
                    + frame['mut_aa'].astype(str).str.upper()
                )
            return frame, str(candidate), ('strict' if 'strict' in candidate.name else 'broad')
    return pd.DataFrame(columns=[
        'variant_id',
        'excess_local_rmsd_median',
        'contact_change_fraction_median',
        'pair_rank_norm',
        'll_rank_norm',
    ]), 'none', 'missing'

def parse_fasta_sequence(path: Path) -> str:
    lines = [line.strip() for line in path.read_text(encoding='utf-8').splitlines() if line.strip() and not line.startswith('>')]
    return ''.join(lines).upper()

def validate_panel_against_sequence(panel_df: pd.DataFrame, sequence: str, gene: str, panel_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    keep_rows = []
    audit_rows = []
    for row in panel_df.to_dict(orient='records'):
        position = int(row.get('position', -1))
        wt_aa = str(row.get('wt_aa', '')).upper()
        observed = None
        reason = None
        if position < 0 or position >= len(sequence):
            reason = 'position_out_of_range'
        else:
            observed = sequence[position].upper()
            if observed != wt_aa:
                reason = 'wt_mismatch'
        if reason is None:
            keep_rows.append(row)
        else:
            audit_rows.append({
                'panel_name': panel_name,
                'gene': gene,
                'variant_id': str(row.get('variant_id', row.get('name', ''))),
                'name': str(row.get('name', '')),
                'position': position,
                'wt_aa': wt_aa,
                'mut_aa': str(row.get('mut_aa', '')).upper(),
                'reason': reason,
                'observed_residue': observed,
            })
    keep_df = pd.DataFrame(keep_rows)
    if keep_df.empty:
        keep_df = panel_df.iloc[0:0].copy()
    audit_df = pd.DataFrame(audit_rows)
    return keep_df, audit_df

def load_model_bundle(spec: dict) -> tuple[dict | None, dict]:
    manifest = {
        'model_label': spec['model_label'],
        'model_name': spec['model_name'],
        'family_label': spec['family_label'],
        'architecture_kind': spec['architecture_kind'],
        'adapter_kind': spec['adapter_kind'],
    }
    try:
        tokenizer = AutoTokenizer.from_pretrained(spec['model_name'], trust_remote_code=True)
        torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        if spec['adapter_kind'] == 't5':
            model = AutoModelForSeq2SeqLM.from_pretrained(spec['model_name'], torch_dtype=torch_dtype, trust_remote_code=True)
        elif spec['adapter_kind'] == 'causal':
            model = AutoModel.from_pretrained(spec['model_name'], torch_dtype=torch_dtype, trust_remote_code=True)
        elif spec['adapter_kind'] == 'auto':
            try:
                model = AutoModel.from_pretrained(spec['model_name'], torch_dtype=torch_dtype, trust_remote_code=True)
            except Exception:
                model = AutoModelForMaskedLM.from_pretrained(spec['model_name'], torch_dtype=torch_dtype, trust_remote_code=True)
        else:
            model = AutoModelForMaskedLM.from_pretrained(spec['model_name'], torch_dtype=torch_dtype, trust_remote_code=True)
        model = model.to(device)
        model.eval()
        manifest.update({
            'status': 'loaded',
            'device': device,
            'torch_dtype': str(torch_dtype),
        })
        return {'tokenizer': tokenizer, 'model': model, 'device': device}, manifest
    except Exception as exc:
        manifest.update({
            'status': 'load_failed',
            'error_type': type(exc).__name__,
            'error_message': str(exc)[:500],
        })
        return None, manifest

def encode_sequence(bundle: dict, sequence: str, sequence_mode: str) -> dict[str, torch.Tensor]:
    encoded = bundle['tokenizer'](
        normalize_protein_sequence(sequence, sequence_mode),
        return_tensors='pt',
        add_special_tokens=False,
    )
    return {key: value.to(bundle['device']) for key, value in encoded.items()}

def stack_hidden_states(outputs, residue_count: int) -> np.ndarray:
    hidden_states = getattr(outputs, 'hidden_states', None)
    if hidden_states is None:
        hidden_states = getattr(outputs, 'encoder_hidden_states', None)
    if hidden_states is None:
        raise RuntimeError('Model output does not expose hidden states.')
    selected = select_hidden_layers(hidden_states)
    layers = []
    for layer in selected:
        layers.append(layer[0, :residue_count, :].detach().cpu().float().numpy())
    return np.stack(layers, axis=0)

def generic_hidden_forward(bundle: dict, spec: dict, encoded: dict[str, torch.Tensor]):
    model = bundle['model']
    if spec['adapter_kind'] == 'causal':
        return model(**encoded, output_hidden_states=True, use_cache=False)
    return model(**encoded, output_hidden_states=True)

def score_panel_with_generic_model(panel_df: pd.DataFrame, sequence_map: dict[str, str], spec: dict, output_path: Path) -> tuple[pd.DataFrame, dict]:
    if output_path.exists() and not OVERWRITE:
        reused = pd.read_csv(output_path)
        return reused, {
            'model_label': spec['model_label'],
            'model_name': spec['model_name'],
            'status': 'reused_existing_scores',
            'n_rows': int(len(reused)),
        }

    bundle, manifest = load_model_bundle(spec)
    if bundle is None:
        return pd.DataFrame(), manifest

    rows = []
    wt_cache: dict[tuple[str, int], np.ndarray] = {}
    try:
        for index, row in enumerate(panel_df.to_dict(orient='records'), start=1):
            gene = str(row['gene']).upper()
            sequence = sequence_map[gene]
            position = int(row['position'])
            wt_aa = str(row['wt_aa']).upper()
            mut_aa = str(row['mut_aa']).upper()
            if sequence[position].upper() != wt_aa:
                raise ValueError(f'WT mismatch for {row["variant_id"]}: expected {wt_aa}, found {sequence[position]}')

            start = max(0, position - WINDOW_RADIUS)
            end = min(len(sequence), position + WINDOW_RADIUS + 1)
            local_sequence = sequence[start:end]
            local_pos = position - start
            wt_key = (gene, position)

            if wt_key not in wt_cache:
                wt_encoded = encode_sequence(bundle, local_sequence, spec['sequence_mode'])
                token_count = int(wt_encoded['input_ids'].shape[1])
                if token_count != len(local_sequence):
                    raise RuntimeError(
                        f'Tokenization mismatch for {spec["model_label"]}: expected {len(local_sequence)} residues, got {token_count} tokens'
                    )
                with torch.inference_mode():
                    wt_outputs = generic_hidden_forward(bundle, spec, wt_encoded)
                wt_cache[wt_key] = stack_hidden_states(wt_outputs, len(local_sequence))

            wt_hidden = wt_cache[wt_key]
            mutated_local = local_sequence[:local_pos] + mut_aa + local_sequence[local_pos + 1:]
            mut_encoded = encode_sequence(bundle, mutated_local, spec['sequence_mode'])
            token_count = int(mut_encoded['input_ids'].shape[1])
            if token_count != len(mutated_local):
                raise RuntimeError(
                    f'Mut tokenization mismatch for {spec["model_label"]}: expected {len(mutated_local)} residues, got {token_count} tokens'
                )
            with torch.inference_mode():
                mut_outputs = generic_hidden_forward(bundle, spec, mut_encoded)
            mut_hidden = stack_hidden_states(mut_outputs, len(mutated_local))
            cov = covariance_features_dual(wt_hidden, mut_hidden)
            token_shift = float(np.linalg.norm(mut_hidden[-1][local_pos] - wt_hidden[-1][local_pos]))
            rows.append({
                'gene': gene,
                'name': row['name'],
                'position': position,
                'wt_aa': wt_aa,
                'mut_aa': mut_aa,
                'label': int(row['label']),
                'variant_id': row['variant_id'],
                'frob_dist': float(cov['frob_dist']),
                'trace_ratio': float(cov['trace_ratio']),
                'sps_log': float(cov['sps_log']),
                'll_proper': token_shift,
                'model_name': spec['model_name'],
            })
            if index % CHECKPOINT_EVERY == 0 or index == len(panel_df):
                pd.DataFrame(rows).to_csv(output_path, index=False)
        output = pd.DataFrame(rows)
        output.to_csv(output_path, index=False)
        manifest.update({'status': 'completed', 'n_rows': int(len(output))})
        return output, manifest
    except Exception as exc:
        partial = pd.DataFrame(rows)
        if not partial.empty:
            partial.to_csv(output_path, index=False)
        manifest.update({
            'status': 'partial_failure' if not partial.empty else 'scoring_failed',
            'n_rows': int(len(partial)),
            'error_type': type(exc).__name__,
            'error_message': str(exc)[:500],
        })
        return partial, manifest
    finally:
        try:
            del bundle['model']
        except Exception:
            pass
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def score_panel_with_model(panel_df: pd.DataFrame, sequence_map: dict[str, str], spec: dict, output_path: Path) -> tuple[pd.DataFrame, dict]:
    if spec['adapter_kind'] == 'esm':
        if output_path.exists() and not OVERWRITE:
            reused = pd.read_csv(output_path)
            return reused, {
                'model_label': spec['model_label'],
                'model_name': spec['model_name'],
                'status': 'reused_existing_scores',
                'n_rows': int(len(reused)),
                'adapter_kind': 'esm',
            }
        grouped_rows = []
        for gene, gene_df in panel_df.groupby('gene', sort=False):
            gene_key = str(gene).upper()
            if gene_key not in sequence_map:
                raise KeyError(f'Missing sequence for gene {gene_key} in panel scoring.')
            gene_rows = _ensure_gene_score_rows(
                gene=gene_key,
                sequence=sequence_map[gene_key],
                variants=gene_df[['gene', 'name', 'position', 'wt_aa', 'mut_aa', 'label', 'variant_id']].to_dict(orient='records'),
                model_name=spec['model_name'],
                output_dir=output_path.parent,
                window_radius=WINDOW_RADIUS,
                checkpoint_every=CHECKPOINT_EVERY,
                overwrite=OVERWRITE,
            )
            grouped_rows.extend(gene_rows)
        output = pd.DataFrame(grouped_rows)
        if not output.empty and 'variant_id' in output.columns and 'variant_id' in panel_df.columns:
            ordering = panel_df[['variant_id']].copy()
            ordering['_order'] = np.arange(len(ordering))
            output = output.merge(ordering, on='variant_id', how='left')
            output = output.sort_values(['_order', 'variant_id'], kind='stable').drop(columns=['_order'])
        output.to_csv(output_path, index=False)
        return output, {
            'model_label': spec['model_label'],
            'model_name': spec['model_name'],
            'status': 'completed',
            'n_rows': int(len(output)),
            'adapter_kind': 'esm',
        }
    return score_panel_with_generic_model(panel_df, sequence_map, spec, output_path)

runtime_manifest = {
    'repo_root': str(REPO_ROOT),
    'head_commit': run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT),
    'branch': run(['git', 'branch', '--show-current'], cwd=REPO_ROOT),
    'cuda_available': bool(torch.cuda.is_available()),
    'cuda_device_count': int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    'cuda_device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
    'rerun_brca1': RERUN_BRCA1,
    'include_ankh': INCLUDE_ANKH,
}
(RUNTIME_DIR / 'runtime_manifest.json').write_text(json.dumps(runtime_manifest, indent=2), encoding='utf-8')
display(pd.DataFrame([runtime_manifest]))
done('Helpers, repo wiring, statistics, structural loaders, and live scoring adapters are ready.')

print("TERMINEI PODE SEGUIR")


C:\Users\Davib\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,repo_root,head_commit,branch,cuda_available,cuda_device_count,cuda_device_name,rerun_brca1,include_ankh
0,C:\Users\Davib\OneDrive\Área de Trabalho\Stanf...,036b4cf03784ceaa7ec4e9b76bff9c35c10ed0e5,codex/claw4s-rebuild,False,0,cpu,False,False


Helpers, repo wiring, statistics, structural loaders, and live scoring adapters are ready.
TERMINEI PODE SEGUIR
TERMINEI PODE SEGUIR


In [3]:
# Load Block 12B and 12C artifacts, then define localized witness panels and final adjudication helpers
BLOCK12B_ROOT = RESULTS_DIR / BLOCK12B_SLUG
BLOCK12B_TABLES_DIR = BLOCK12B_ROOT / 'tables'
BLOCK12C_SLUG = '12c_block12_covariance_adjudication_structural_closure_h100'
BLOCK12C_ROOT_CANDIDATES = [
    RESULTS_DIR / BLOCK12C_SLUG,
    RESULTS_DIR / f'{BLOCK12C_SLUG}_bundle',
]
BLOCK12B_LIVE_SCORES_DIR = BLOCK12B_ROOT / 'live_scores'

LOCAL_MIN_RULE_ON_ABS = int(os.environ.get('SPECTRALBIO_BLOCK12D_LOCAL_MIN_RULE_ON_ABS', '6'))
LOCAL_MIN_RULE_ON_FRAC = float(os.environ.get('SPECTRALBIO_BLOCK12D_LOCAL_MIN_RULE_ON_FRAC', '0.08'))
PAIR_THRESHOLDS = [
    float(item.strip())
    for item in os.environ.get('SPECTRALBIO_BLOCK12D_PAIR_THRESHOLDS', '0.05,0.10,0.15,0.20,0.25,0.30').split(',')
    if item.strip()
]

required_12b_tables = {
    'variant_scores': BLOCK12B_TABLES_DIR / 'tp53_model_variant_scores.csv',
    'model_summary': BLOCK12B_TABLES_DIR / 'tp53_model_summary.csv',
    'calibration': BLOCK12B_TABLES_DIR / 'tp53_rule_candidate_calibration.csv',
    'selected_rules': BLOCK12B_TABLES_DIR / 'tp53_selected_rules.csv',
    'controls': BLOCK12B_TABLES_DIR / 'tp53_rule_vs_controls.csv',
    'same_position': BLOCK12B_TABLES_DIR / 'tp53_same_position_ranking_summary.csv',
    'robustness': BLOCK12B_TABLES_DIR / 'tp53_robustness_slice_summary.csv',
    'failure': BLOCK12B_TABLES_DIR / 'tp53_failure_taxonomy.csv',
}
missing_12b = [name for name, path in required_12b_tables.items() if not path.exists()]
if missing_12b:
    raise FileNotFoundError(f'Missing required Block 12B tables: {missing_12b}')

def resolve_block12c_tables() -> tuple[dict[str, Path] | None, Path | None]:
    for root in BLOCK12C_ROOT_CANDIDATES:
        tables_dir = root / 'tables'
        manifests_dir = root / 'manifests'
        required = {
            'adjudication': tables_dir / 'tp53_covariance_adjudication_summary.csv',
            'structural': tables_dir / 'tp53_structural_closure_summary.csv',
            'artifact_summary': manifests_dir / 'artifact_summary.json',
        }
        if all(path.exists() for path in required.values()):
            return required, root
    return None, None

required_12c_tables, resolved_block12c_root = resolve_block12c_tables()
missing_12c = [] if required_12c_tables is not None else ['adjudication', 'structural', 'artifact_summary']
if missing_12c:
    raise FileNotFoundError(
        f'Missing required Block 12C outputs: {missing_12c}. '
        f'Looked in {[str(path) for path in BLOCK12C_ROOT_CANDIDATES]}. '
        f'Run {BLOCK12C_SLUG} before Block 12D.'
    )
BLOCK12C_TABLES_DIR = resolved_block12c_root / 'tables'
BLOCK12C_MANIFESTS_DIR = resolved_block12c_root / 'manifests'

variant_scores_12b = pd.read_csv(required_12b_tables['variant_scores'])
model_summary_12b = pd.read_csv(required_12b_tables['model_summary'])
calibration_12b = pd.read_csv(required_12b_tables['calibration'])
selected_rules_12b = pd.read_csv(required_12b_tables['selected_rules'])
control_table_12b = pd.read_csv(required_12b_tables['controls'])
same_position_12b = pd.read_csv(required_12b_tables['same_position'])
robustness_12b = pd.read_csv(required_12b_tables['robustness'])
failure_12b = pd.read_csv(required_12b_tables['failure'])
covariance_12c = pd.read_csv(required_12c_tables['adjudication'])
structural_12c = pd.read_csv(required_12c_tables['structural'])

block12c_artifact_summary = json.loads(required_12c_tables['artifact_summary'].read_text(encoding='utf-8'))

structural_reference, structural_source_path, structural_source_kind = load_structural_reference()
structural_positions_zero_based = sorted(
    {
        int(value) - 1
        for value in pd.to_numeric(structural_reference.get('position_1', pd.Series(dtype=float)), errors='coerce').dropna().astype(int).tolist()
    }
)
structural_variant_ids = set(structural_reference.get('variant_id', pd.Series(dtype=str)).astype(str).tolist())

variant_scores_12b = variant_scores_12b.copy()
if 'pair_rank_fixed_055' not in variant_scores_12b.columns:
    variant_scores_12b['pair_rank_fixed_055'] = (
        FIXED_ALPHA * pd.to_numeric(variant_scores_12b['frob_rank_norm'], errors='coerce').fillna(0.0)
        + (1.0 - FIXED_ALPHA) * pd.to_numeric(variant_scores_12b['ll_rank_norm'], errors='coerce').fillna(0.0)
    )
if 'pair_minus_ll' not in variant_scores_12b.columns:
    variant_scores_12b['pair_minus_ll'] = (
        pd.to_numeric(variant_scores_12b['pair_rank_fixed_055'], errors='coerce').fillna(0.0)
        - pd.to_numeric(variant_scores_12b['ll_rank_norm'], errors='coerce').fillna(0.0)
    )
variant_scores_12b['tp53_domain'] = variant_scores_12b['position'].astype(int).map(tp53_domain_label)
variant_scores_12b['panel_full_tp53'] = True
variant_scores_12b['panel_dna_binding_core'] = variant_scores_12b['tp53_domain'].eq('dna_binding')
variant_scores_12b['panel_hotspot_core'] = variant_scores_12b['tp53_hotspot'].fillna(False).astype(bool)
variant_scores_12b['panel_dna_binding_plus_hotspot'] = (
    variant_scores_12b['panel_dna_binding_core'] | variant_scores_12b['panel_hotspot_core']
)
variant_scores_12b['panel_structural_positions'] = variant_scores_12b['position'].astype(int).isin(structural_positions_zero_based)
variant_scores_12b['panel_localized_union'] = (
    variant_scores_12b['panel_dna_binding_core']
    | variant_scores_12b['panel_hotspot_core']
    | variant_scores_12b['panel_structural_positions']
)

target_panel_specs = [
    {
        'panel_name': 'full_tp53',
        'panel_column': 'panel_full_tp53',
        'scope_label': 'global',
        'preferred_same_position_mode': 'global',
    },
    {
        'panel_name': 'dna_binding_core',
        'panel_column': 'panel_dna_binding_core',
        'scope_label': 'localized',
        'preferred_same_position_mode': 'selected_positions',
    },
    {
        'panel_name': 'dna_binding_plus_hotspot',
        'panel_column': 'panel_dna_binding_plus_hotspot',
        'scope_label': 'localized',
        'preferred_same_position_mode': 'selected_positions',
    },
    {
        'panel_name': 'localized_union',
        'panel_column': 'panel_localized_union',
        'scope_label': 'localized',
        'preferred_same_position_mode': 'selected_positions',
    },
]

def local_coverage_floor(n_variants: int) -> int:
    return max(LOCAL_MIN_RULE_ON_ABS, int(math.ceil(float(n_variants) * LOCAL_MIN_RULE_ON_FRAC)))

def top_n_mask_from_scores(frame: pd.DataFrame, score_column: str, n: int) -> pd.Series:
    if n <= 0 or frame.empty:
        return pd.Series(np.zeros(len(frame), dtype=bool), index=frame.index)
    ranked = pd.to_numeric(frame[score_column], errors='coerce').fillna(-999.0)
    top_index = ranked.sort_values(ascending=False).index[: min(n, len(frame))]
    return pd.Series(frame.index.isin(top_index), index=frame.index)

def enrichment_gap_for_mask(frame: pd.DataFrame, mask: pd.Series, seed: int) -> dict:
    mask_series = pd.Series(mask, index=frame.index).fillna(False).astype(bool)
    labels = frame['label'].astype(int).to_numpy()
    gap_stats = bootstrap_gap(labels, mask_series.to_numpy(), BOOTSTRAP_REPLICATES, seed)
    odds = odds_ratio_from_mask(labels, mask_series.to_numpy())
    return {
        **gap_stats,
        'odds_ratio': odds['odds_ratio'],
        'positive_with_rule': odds['positive_with_rule'],
        'negative_with_rule': odds['negative_with_rule'],
        'positive_without_rule': odds['positive_without_rule'],
        'negative_without_rule': odds['negative_without_rule'],
    }

def chemistry_gap_for_panel(frame: pd.DataFrame, seed: int) -> dict:
    chemistry_mask = frame['chemistry_trigger'].fillna(False).astype(bool)
    if chemistry_mask.sum() == 0 or (~chemistry_mask).sum() == 0:
        return {'enrichment_gap': float('nan')}
    return enrichment_gap_for_mask(frame, chemistry_mask, seed + 701)

def same_position_witness(frame: pd.DataFrame, candidate_mask: pd.Series, mode: str) -> dict:
    if candidate_mask.sum() == 0:
        return {
            'verdict': 'not_applicable',
            'n_mixed_positions': 0,
            'pair_top_hit_rate': float('nan'),
            'll_top_hit_rate': float('nan'),
            'delta': float('nan'),
        }
    if mode == 'selected_positions':
        working = frame.loc[frame['position'].isin(frame.loc[candidate_mask, 'position'].astype(int).tolist())].copy()
    else:
        working = frame.copy()

    pair_hits = []
    ll_hits = []
    mixed_positions = 0
    for _, group in working.groupby('position'):
        labels = group['label'].astype(int)
        if labels.nunique() < 2:
            continue
        pair_values = pd.to_numeric(group['pair_minus_ll'], errors='coerce')
        ll_values = pd.to_numeric(group['ll_rank_norm'], errors='coerce')
        if pair_values.notna().sum() == 0 or ll_values.notna().sum() == 0:
            continue
        pair_idx = pair_values.idxmax()
        ll_idx = ll_values.idxmax()
        pair_hits.append(int(group.loc[pair_idx, 'label']))
        ll_hits.append(int(group.loc[ll_idx, 'label']))
        mixed_positions += 1

    if mixed_positions == 0:
        return {
            'verdict': 'not_applicable',
            'n_mixed_positions': 0,
            'pair_top_hit_rate': float('nan'),
            'll_top_hit_rate': float('nan'),
            'delta': float('nan'),
        }

    pair_rate = float(np.mean(pair_hits))
    ll_rate = float(np.mean(ll_hits))
    delta = pair_rate - ll_rate
    if delta > 1e-9:
        verdict = 'win'
    elif delta < -1e-9:
        verdict = 'lose'
    else:
        verdict = 'tie'
    return {
        'verdict': verdict,
        'n_mixed_positions': mixed_positions,
        'pair_top_hit_rate': pair_rate,
        'll_top_hit_rate': ll_rate,
        'delta': delta,
    }

structural_summary_by_model = structural_12c.set_index('model_label').to_dict(orient='index')

def structural_witness_for_model(model_label: str) -> dict:
    row = structural_summary_by_model.get(model_label, {})
    pair_spearman = safe_float(row.get('spearman_pair_minus_ll_vs_excess_local_rmsd'), float('nan'))
    ll_spearman = safe_float(row.get('spearman_ll_rank_vs_excess_local_rmsd'), float('nan'))
    geometry_gap = safe_float(row.get('structural_geometry_gap'), float('nan'))
    positive = False
    if math.isfinite(pair_spearman) and math.isfinite(ll_spearman) and pair_spearman > ll_spearman:
        positive = True
    if math.isfinite(geometry_gap) and geometry_gap > 0:
        positive = True
    if positive:
        verdict = 'positive'
    elif math.isfinite(pair_spearman) or math.isfinite(ll_spearman) or math.isfinite(geometry_gap):
        verdict = 'negative'
    else:
        verdict = 'missing'
    return {
        'verdict': verdict,
        'pair_spearman': pair_spearman,
        'll_spearman': ll_spearman,
        'geometry_gap': geometry_gap,
        'n_structural_rows': int(row.get('n_structural_rows', 0)) if row else 0,
    }

model_context_rows = []
covariance_context = covariance_12c.set_index('model_label').to_dict(orient='index')
selected_rule_context = selected_rules_12b.set_index('model_label').to_dict(orient='index')
for spec in MODEL_SPECS:
    model_label = spec['model_label']
    structural_witness = structural_witness_for_model(model_label)
    covariance_row = covariance_context.get(model_label, {})
    selected_row = selected_rule_context.get(model_label, {})
    model_context_rows.append({
        'model_label': model_label,
        'family_label': spec['family_label'],
        'architecture_kind': spec['architecture_kind'],
        'block12c_status': str(covariance_row.get('adjudication_status', 'missing')),
        'block12c_reason': str(covariance_row.get('adjudication_reason', 'missing')),
        'block12b_selected_rule_type': str(selected_row.get('rule_type', 'missing')),
        'block12b_selected_gap': safe_float(selected_row.get('enrichment_gap'), float('nan')),
        'structural_witness_verdict': structural_witness['verdict'],
        'structural_pair_spearman': structural_witness['pair_spearman'],
        'structural_ll_spearman': structural_witness['ll_spearman'],
        'structural_geometry_gap': structural_witness['geometry_gap'],
        'structural_rows': structural_witness['n_structural_rows'],
    })

model_context_table = pd.DataFrame(model_context_rows).sort_values(
    ['structural_witness_verdict', 'block12c_status', 'model_label'],
    ascending=[True, True, True],
).reset_index(drop=True)

model_context_table.to_csv(TABLES_DIR / 'tp53_block12d_model_context.csv', index=False)
display(model_context_table)
done('Block 12B and 12C artifacts are loaded, localized witness panels are defined, and the final search helpers are ready.')

print("TERMINEI PODE SEGUIR")


,model_label,family_label,architecture_kind,block12c_status,block12c_reason,block12b_selected_rule_type,block12b_selected_gap,structural_witness_verdict,structural_pair_spearman,structural_ll_spearman,structural_geometry_gap,structural_rows
0,ESM2-35M,esm,masked_encoder,not_eligible,coverage_or_enrichment_failure,scalar_only,0.461847,negative,0.0,0.0,NaN,4
1,ProtBERT-BFD,bert_style,bidirectional_encoder,not_eligible,coverage_or_enrichment_failure,scalar_only,0.463710,negative,-1.0,0.8,NaN,4
2,ESM2-150M,esm,masked_encoder,supportive,positive_but_not_fully_adjudicated,pair_only,0.319038,negative,-0.4,0.4,NaN,4
3,ESM2-3B,esm,masked_encoder,supportive,positive_but_not_fully_adjudicated,scalar_only,0.454545,negative,0.0,0.0,NaN,4
4,ProGen2-base,progen,causal_decoder,supportive,positive_but_not_fully_adjudicated,scalar_only,0.467480,negative,-0.6,1.0,NaN,4
5,ProtBERT,bert_style,bidirectional_encoder,not_eligible,coverage_or_enrichment_failure,chemistry_only,0.086077,positive,0.2,-0.4,NaN,4
6,ProtT5,prottrans,encoder_decoder,not_eligible,coverage_or_enrichment_failure,scalar_only,0.204183,positive,-0.2,-0.8,NaN,4
7,ESM-1v,esm_variant_specialist,masked_encoder,supportive,positive_but_not_fully_adjudicated,scalar_only,0.454545,positive,0.4,-0.2,NaN,4
8,ESM2-650M,esm,masked_encoder,supportive,positive_but_not_fully_adjudicated,pair_only,0.203023,positive,-0.4,0.4,0.333333,4
9,ProGen2-small,progen,causal_decoder,supportive,positive_but_not_fully_adjudicated,pair_only,0.512500,positive,1.0,-0.4,0.333333,4


Block 12B and 12C artifacts are loaded, localized witness panels are defined, and the final search helpers are ready.
TERMINEI PODE SEGUIR
TERMINEI PODE SEGUIR


In [4]:
# Search the final localized rule space and identify whether a bounded nuclear regime exists
candidate_rows = []

rule_templates = [
    {
        'rule_family': 'pair_only',
        'threshold_source': 'pair_minus_ll',
        'gate_name': 'none',
        'gate_fn': lambda frame: pd.Series(np.ones(len(frame), dtype=bool), index=frame.index),
    },
    {
        'rule_family': 'pair_only',
        'threshold_source': 'pair_minus_ll',
        'gate_name': 'exclude_chemistry',
        'gate_fn': lambda frame: ~frame['chemistry_trigger'].fillna(False).astype(bool),
    },
    {
        'rule_family': 'pair_only',
        'threshold_source': 'pair_minus_ll',
        'gate_name': 'structural_or_hotspot',
        'gate_fn': lambda frame: (
            frame['panel_structural_positions'].fillna(False).astype(bool)
            | frame['panel_hotspot_core'].fillna(False).astype(bool)
        ),
    },
    {
        'rule_family': 'pair_rank_fixed',
        'threshold_source': 'pair_rank_fixed_055',
        'gate_name': 'none',
        'gate_fn': lambda frame: pd.Series(np.ones(len(frame), dtype=bool), index=frame.index),
    },
]

for spec_index, spec in enumerate(MODEL_SPECS):
    model_label = spec['model_label']
    model_frame = variant_scores_12b.loc[variant_scores_12b['model_label'].eq(model_label)].copy().reset_index(drop=True)
    if model_frame.empty:
        continue

    labels_full = model_frame['label'].astype(int).to_numpy()
    structural_witness = structural_witness_for_model(model_label)

    for panel_index, panel_spec in enumerate(target_panel_specs):
        panel_mask = model_frame[panel_spec['panel_column']].fillna(False).astype(bool)
        panel_frame = model_frame.loc[panel_mask].copy().reset_index(drop=True)
        if panel_frame.empty:
            continue
        panel_size = int(len(panel_frame))
        coverage_target = local_coverage_floor(panel_size) if panel_spec['scope_label'] == 'localized' else coverage_floor(panel_size)
        chemistry_local = chemistry_gap_for_panel(panel_frame, RANDOM_SEED + spec_index * 1000 + panel_index * 100)

        for template_index, template in enumerate(rule_templates):
            threshold_values = PAIR_THRESHOLDS if template['threshold_source'] != 'pair_rank_fixed_055' else [0.55, 0.60, 0.65, 0.70, 0.75]
            base_gate = template['gate_fn'](panel_frame).fillna(False).astype(bool)

            for threshold in threshold_values:
                score_values = pd.to_numeric(panel_frame[template['threshold_source']], errors='coerce').fillna(-999.0)
                candidate_mask = base_gate & score_values.ge(float(threshold))
                n_rule_on = int(candidate_mask.sum())
                if n_rule_on <= 0 or n_rule_on >= len(panel_frame):
                    continue

                gap_stats = enrichment_gap_for_mask(
                    panel_frame,
                    candidate_mask,
                    RANDOM_SEED + spec_index * 1000 + panel_index * 100 + template_index * 10 + int(round(float(threshold) * 100)),
                )
                global_scalar_gap = enrichment_gap_for_mask(
                    model_frame,
                    top_n_mask_from_scores(model_frame, 'll_rank_norm', n_rule_on),
                    RANDOM_SEED + 5000 + spec_index * 100 + n_rule_on,
                )['enrichment_gap']
                local_scalar_gap = enrichment_gap_for_mask(
                    panel_frame,
                    top_n_mask_from_scores(panel_frame, 'll_rank_norm', n_rule_on),
                    RANDOM_SEED + 6000 + panel_index * 100 + n_rule_on,
                )['enrichment_gap']
                local_pair_baseline_gap = enrichment_gap_for_mask(
                    panel_frame,
                    top_n_mask_from_scores(panel_frame, template['threshold_source'], n_rule_on),
                    RANDOM_SEED + 7000 + template_index * 100 + n_rule_on,
                )['enrichment_gap']
                same_position = same_position_witness(
                    panel_frame,
                    candidate_mask,
                    panel_spec['preferred_same_position_mode'],
                )

                beats_global_scalar = (
                    math.isfinite(gap_stats['enrichment_gap'])
                    and math.isfinite(global_scalar_gap)
                    and gap_stats['enrichment_gap'] > global_scalar_gap
                )
                beats_local_scalar = (
                    math.isfinite(gap_stats['enrichment_gap'])
                    and math.isfinite(local_scalar_gap)
                    and gap_stats['enrichment_gap'] > local_scalar_gap
                )
                beats_local_chemistry = (
                    math.isfinite(gap_stats['enrichment_gap'])
                    and math.isfinite(chemistry_local['enrichment_gap'])
                    and gap_stats['enrichment_gap'] > chemistry_local['enrichment_gap']
                )
                same_position_ok = same_position['verdict'] in {'win', 'tie', 'not_applicable'}
                structural_ok = structural_witness['verdict'] == 'positive'
                coverage_pass = bool(n_rule_on >= coverage_target)
                gap_positive = math.isfinite(gap_stats['enrichment_gap']) and gap_stats['enrichment_gap'] > 0

                if (
                    panel_spec['scope_label'] == 'localized'
                    and coverage_pass
                    and gap_positive
                    and beats_global_scalar
                    and beats_local_scalar
                    and (beats_local_chemistry or not math.isfinite(chemistry_local['enrichment_gap']))
                    and same_position_ok
                    and structural_ok
                    and template['rule_family'] == 'pair_only'
                ):
                    adjudication_status = 'nuclear_localized'
                    adjudication_reason = 'localized_pair_rule_beats_global_and_local_scalar_with_structural_witness'
                elif (
                    coverage_pass
                    and gap_positive
                    and beats_local_scalar
                    and same_position_ok
                ):
                    adjudication_status = 'supportive_localized'
                    adjudication_reason = 'localized_rule_positive_but_not_fully_nuclear'
                elif coverage_pass and gap_positive:
                    adjudication_status = 'supportive_residual'
                    adjudication_reason = 'positive_gap_without_full_control_dominance'
                else:
                    adjudication_status = 'not_eligible'
                    adjudication_reason = 'coverage_or_control_failure'

                candidate_rows.append({
                    'model_label': model_label,
                    'family_label': spec['family_label'],
                    'architecture_kind': spec['architecture_kind'],
                    'panel_name': panel_spec['panel_name'],
                    'scope_label': panel_spec['scope_label'],
                    'panel_size': panel_size,
                    'coverage_target': coverage_target,
                    'coverage_pass': coverage_pass,
                    'rule_family': template['rule_family'],
                    'gate_name': template['gate_name'],
                    'threshold_source': template['threshold_source'],
                    'threshold': float(threshold),
                    'n_rule_on': n_rule_on,
                    'fraction_rule_on': float(n_rule_on / float(panel_size)),
                    'enrichment_gap': gap_stats['enrichment_gap'],
                    'gap_ci_low': gap_stats['gap_ci_low'],
                    'gap_ci_high': gap_stats['gap_ci_high'],
                    'odds_ratio': gap_stats['odds_ratio'],
                    'global_scalar_gap': global_scalar_gap,
                    'local_scalar_gap': local_scalar_gap,
                    'local_pair_baseline_gap': local_pair_baseline_gap,
                    'local_chemistry_gap': chemistry_local['enrichment_gap'],
                    'beats_global_scalar': bool(beats_global_scalar),
                    'beats_local_scalar': bool(beats_local_scalar),
                    'beats_local_chemistry': bool(beats_local_chemistry) if math.isfinite(chemistry_local['enrichment_gap']) else True,
                    'same_position_verdict': same_position['verdict'],
                    'same_position_n_mixed_positions': same_position['n_mixed_positions'],
                    'same_position_pair_top_hit_rate': same_position['pair_top_hit_rate'],
                    'same_position_ll_top_hit_rate': same_position['ll_top_hit_rate'],
                    'same_position_delta': same_position['delta'],
                    'structural_witness_verdict': structural_witness['verdict'],
                    'structural_pair_spearman': structural_witness['pair_spearman'],
                    'structural_ll_spearman': structural_witness['ll_spearman'],
                    'structural_geometry_gap': structural_witness['geometry_gap'],
                    'block12c_status': covariance_context.get(model_label, {}).get('adjudication_status', 'missing'),
                    'adjudication_status': adjudication_status,
                    'adjudication_reason': adjudication_reason,
                })

localized_rule_search = pd.DataFrame(candidate_rows)
if localized_rule_search.empty:
    raise RuntimeError('Block 12D did not generate any candidate rules.')

status_rank = {
    'nuclear_localized': 0,
    'supportive_localized': 1,
    'supportive_residual': 2,
    'not_eligible': 3,
}
localized_rule_search['status_rank'] = localized_rule_search['adjudication_status'].map(status_rank).fillna(9).astype(int)
localized_rule_search['same_position_rank'] = localized_rule_search['same_position_verdict'].map(
    {'win': 0, 'tie': 1, 'not_applicable': 2, 'lose': 3}
).fillna(4).astype(int)

localized_best_rules = localized_rule_search.sort_values(
    [
        'status_rank',
        'same_position_rank',
        'enrichment_gap',
        'structural_geometry_gap',
        'n_rule_on',
    ],
    ascending=[True, True, False, False, True],
).groupby('model_label', as_index=False).first()

nuclear_localized = localized_best_rules.loc[
    localized_best_rules['adjudication_status'].eq('nuclear_localized')
].copy()
supportive_localized = localized_best_rules.loc[
    localized_best_rules['adjudication_status'].eq('supportive_localized')
].copy()

localized_rule_search.to_csv(TABLES_DIR / 'tp53_localized_rule_search.csv', index=False)
localized_best_rules.to_csv(TABLES_DIR / 'tp53_localized_best_rules.csv', index=False)
nuclear_localized.to_csv(TABLES_DIR / 'tp53_localized_nuclear_candidates.csv', index=False)
supportive_localized.to_csv(TABLES_DIR / 'tp53_localized_supportive_candidates.csv', index=False)

display(localized_best_rules)
display(nuclear_localized if not nuclear_localized.empty else pd.DataFrame([{'status': 'no_nuclear_localized_rule_found_yet'}]))
done('The final localized rule space has been searched and the best per-model candidates are now adjudicated.')

print("TERMINEI PODE SEGUIR")


,model_label,family_label,architecture_kind,panel_name,scope_label,panel_size,coverage_target,coverage_pass,rule_family,gate_name,...,same_position_delta,structural_witness_verdict,structural_pair_spearman,structural_ll_spearman,structural_geometry_gap,block12c_status,adjudication_status,adjudication_reason,status_rank,same_position_rank
0,ESM-1v,esm_variant_specialist,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,0.272727,positive,0.4,-0.2,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0
1,ESM2-150M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,0.181818,negative,-0.4,0.4,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0
2,ESM2-35M,esm,masked_encoder,full_tp53,global,255,13,False,pair_only,none,...,0.272727,negative,0.0,0.0,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,0
3,ESM2-3B,esm,masked_encoder,dna_binding_core,localized,145,12,True,pair_rank_fixed,none,...,0.500000,negative,0.0,0.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0
4,ESM2-650M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,none,...,0.363636,positive,-0.4,0.4,0.333333,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0
5,ProGen2-base,progen,causal_decoder,dna_binding_core,localized,145,12,True,pair_only,none,...,0.333333,negative,-0.6,1.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0
6,ProGen2-small,progen,causal_decoder,full_tp53,global,255,13,True,pair_only,none,...,0.000000,positive,1.0,-0.4,0.333333,supportive,supportive_localized,localized_rule_positive_but_not_fully_nuclear,1,1
7,ProtBERT,bert_style,bidirectional_encoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.272727,positive,0.2,-0.4,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3
8,ProtBERT-BFD,bert_style,bidirectional_encoder,dna_binding_core,localized,145,12,False,pair_rank_fixed,none,...,0.000000,negative,-1.0,0.8,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,1
9,ProtT5,prottrans,encoder_decoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.181818,positive,-0.2,-0.8,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3


,status
0,no_nuclear_localized_rule_found_yet


The final localized rule space has been searched and the best per-model candidates are now adjudicated.
TERMINEI PODE SEGUIR
TERMINEI PODE SEGUIR


In [5]:
# Build transfer witnesses, positive-only external witnesses, and final stress-test summaries
def model_live_score_dir(model_name: str) -> Path:
    return BLOCK12B_LIVE_SCORES_DIR / model_name.replace('/', '_').replace('-', '_')

positive_witness_panels = ['crebbp', 'grin2a', 'tsc2']
transfer_rows = []

for _, best_row in localized_best_rules.iterrows():
    model_label = str(best_row['model_label'])
    model_spec = next((item for item in MODEL_SPECS if item['model_label'] == model_label), None)
    if model_spec is None:
        continue

    threshold_source = str(best_row['threshold_source'])
    threshold = safe_float(best_row['threshold'], float('nan'))
    rule_family = str(best_row['rule_family'])
    if not math.isfinite(threshold):
        continue

    for panel_name in positive_witness_panels:
        score_path = model_live_score_dir(model_spec['model_name']) / f'{panel_name}_{model_spec["model_name"].replace("/", "_").replace("-", "_")}_scores.csv'
        if not score_path.exists():
            transfer_rows.append({
                'model_label': model_label,
                'panel_name': panel_name,
                'n_rows': 0,
                'n_rule_on': 0,
                'hit_fraction': float('nan'),
                'top_pair_variant': 'missing',
                'top_ll_variant': 'missing',
                'transfer_status': 'missing_panel',
            })
            continue

        panel_frame = pd.read_csv(score_path)
        panel_scored = score_vector_from_frame(panel_frame, FIXED_ALPHA)
        gate_mask = pd.Series(np.ones(len(panel_scored), dtype=bool), index=panel_scored.index)
        if rule_family == 'pair_only' and str(best_row['gate_name']) == 'exclude_chemistry':
            gate_mask = ~panel_scored['chemistry_trigger'].fillna(False).astype(bool)
        score_mask = pd.to_numeric(panel_scored[threshold_source], errors='coerce').fillna(-999.0) >= float(threshold)
        rule_mask = gate_mask & score_mask
        top_pair_variant = str(
            panel_scored.loc[pd.to_numeric(panel_scored['pair_minus_ll'], errors='coerce').fillna(-999.0).idxmax(), 'variant_id']
        )
        top_ll_variant = str(
            panel_scored.loc[pd.to_numeric(panel_scored['ll_rank_norm'], errors='coerce').fillna(-999.0).idxmax(), 'variant_id']
        )
        transfer_rows.append({
            'model_label': model_label,
            'panel_name': panel_name,
            'n_rows': int(len(panel_scored)),
            'n_rule_on': int(rule_mask.sum()),
            'hit_fraction': float(rule_mask.mean()) if len(panel_scored) else float('nan'),
            'top_pair_variant': top_pair_variant,
            'top_ll_variant': top_ll_variant,
            'transfer_status': 'present',
        })

transfer_witness_table = pd.DataFrame(transfer_rows)
if transfer_witness_table.empty:
    transfer_witness_summary = pd.DataFrame(columns=['model_label', 'positive_witness_panels_present', 'positive_witness_hit_fraction_mean'])
else:
    transfer_witness_summary = transfer_witness_table.groupby('model_label', as_index=False).agg(
        positive_witness_panels_present=('transfer_status', lambda values: int(sum(item == 'present' for item in values))),
        positive_witness_hit_fraction_mean=('hit_fraction', lambda series: float(pd.to_numeric(series, errors='coerce').mean()) if len(series) else float('nan')),
    )

brca1_transfer_summary_path = BLOCK12C_TABLES_DIR / 'brca1_transfer_adjudication_summary.csv'
brca1_transfer_summary = (
    pd.read_csv(brca1_transfer_summary_path)
    if brca1_transfer_summary_path.exists()
    else pd.DataFrame(columns=['model_label'])
)

final_localized_scoreboard = localized_best_rules.merge(
    transfer_witness_summary,
    on='model_label',
    how='left',
)
if not brca1_transfer_summary.empty and 'model_label' in brca1_transfer_summary.columns:
    brca1_small = brca1_transfer_summary.copy()
    keep_cols = [column for column in brca1_small.columns if column in {'model_label', 'transfer_status', 'transfer_hit_fraction', 'transfer_gap', 'transfer_beats_scalar'}]
    final_localized_scoreboard = final_localized_scoreboard.merge(
        brca1_small[keep_cols],
        on='model_label',
        how='left',
        suffixes=('', '_brca1'),
    )

final_localized_scoreboard['primary_claim_firewall'] = np.where(
    final_localized_scoreboard['adjudication_status'].eq('nuclear_localized'),
    'eligible_for_localized_primary_claim',
    'supportive_or_excluded',
)

transfer_witness_table.to_csv(TABLES_DIR / 'tp53_positive_witness_transfer_table.csv', index=False)
transfer_witness_summary.to_csv(TABLES_DIR / 'tp53_positive_witness_transfer_summary.csv', index=False)
final_localized_scoreboard.to_csv(TABLES_DIR / 'tp53_final_localized_scoreboard.csv', index=False)

display(transfer_witness_summary if not transfer_witness_summary.empty else pd.DataFrame([{'status': 'no_positive_transfer_witnesses_found'}]))
display(final_localized_scoreboard)
done('Positive-only transfer witnesses and the final localized scoreboard are now assembled.')

print("TERMINEI PODE SEGUIR")


,model_label,positive_witness_panels_present,positive_witness_hit_fraction_mean
0,ESM-1v,3,0.000000
1,ESM2-150M,3,0.000000
2,ESM2-35M,3,0.166667
3,ESM2-3B,3,0.833333
4,ESM2-650M,3,0.166667
5,ProGen2-base,0,NaN
6,ProGen2-small,0,NaN
7,ProtBERT,0,NaN
8,ProtBERT-BFD,0,NaN
9,ProtT5,0,NaN


,model_label,family_label,architecture_kind,panel_name,scope_label,panel_size,coverage_target,coverage_pass,rule_family,gate_name,...,structural_ll_spearman,structural_geometry_gap,block12c_status,adjudication_status,adjudication_reason,status_rank,same_position_rank,positive_witness_panels_present,positive_witness_hit_fraction_mean,primary_claim_firewall
0,ESM-1v,esm_variant_specialist,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,-0.2,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.000000,supportive_or_excluded
1,ESM2-150M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,0.4,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.000000,supportive_or_excluded
2,ESM2-35M,esm,masked_encoder,full_tp53,global,255,13,False,pair_only,none,...,0.0,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,0,3,0.166667,supportive_or_excluded
3,ESM2-3B,esm,masked_encoder,dna_binding_core,localized,145,12,True,pair_rank_fixed,none,...,0.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.833333,supportive_or_excluded
4,ESM2-650M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,none,...,0.4,0.333333,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.166667,supportive_or_excluded
5,ProGen2-base,progen,causal_decoder,dna_binding_core,localized,145,12,True,pair_only,none,...,1.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,0,NaN,supportive_or_excluded
6,ProGen2-small,progen,causal_decoder,full_tp53,global,255,13,True,pair_only,none,...,-0.4,0.333333,supportive,supportive_localized,localized_rule_positive_but_not_fully_nuclear,1,1,0,NaN,supportive_or_excluded
7,ProtBERT,bert_style,bidirectional_encoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.4,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3,0,NaN,supportive_or_excluded
8,ProtBERT-BFD,bert_style,bidirectional_encoder,dna_binding_core,localized,145,12,False,pair_rank_fixed,none,...,0.8,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,1,0,NaN,supportive_or_excluded
9,ProtT5,prottrans,encoder_decoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.8,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3,0,NaN,supportive_or_excluded


Positive-only transfer witnesses and the final localized scoreboard are now assembled.
TERMINEI PODE SEGUIR
TERMINEI PODE SEGUIR


In [6]:
# Render figures, write the final manifests, and package the last notebook bundle
plot_rows = final_localized_scoreboard.copy()
plot_rows['model_order'] = np.arange(len(plot_rows))

fig, ax = plt.subplots(figsize=(12, 6))
colors = plot_rows['adjudication_status'].map(
    {
        'nuclear_localized': '#0B6E4F',
        'supportive_localized': '#4C78A8',
        'supportive_residual': '#F28E2B',
        'not_eligible': '#B07AA1',
    }
).fillna('#999999')
ax.barh(plot_rows['model_label'], plot_rows['enrichment_gap'], color=colors)
ax.axvline(0.0, color='black', linewidth=1.0)
ax.set_title('Block 12D localized best-rule enrichment gap by model')
ax.set_xlabel('Localized enrichment gap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'localized_best_rule_scoreboard.png', dpi=200, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(
    localized_rule_search['n_rule_on'],
    localized_rule_search['enrichment_gap'],
    c=localized_rule_search['status_rank'],
    cmap='viridis',
    alpha=0.7,
    s=40,
)
ax.set_title('Block 12D localized rule frontier')
ax.set_xlabel('n rule on')
ax.set_ylabel('localized enrichment gap')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'localized_rule_frontier.png', dpi=200, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots(figsize=(12, 6))
same_position_plot = final_localized_scoreboard[['model_label', 'same_position_delta']].copy()
same_position_plot['same_position_delta'] = pd.to_numeric(same_position_plot['same_position_delta'], errors='coerce')
ax.barh(
    same_position_plot['model_label'],
    same_position_plot['same_position_delta'].fillna(0.0),
    color=['#2C7FB8' if value >= 0 else '#D95F02' for value in same_position_plot['same_position_delta'].fillna(0.0)],
)
ax.axvline(0.0, color='black', linewidth=1.0)
ax.set_title('Same-position witness delta on the selected localized panel')
ax.set_xlabel('pair top-hit rate minus ll top-hit rate')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'localized_same_position_delta.png', dpi=200, bbox_inches='tight')
plt.close(fig)

nuclear_localized = final_localized_scoreboard.loc[
    final_localized_scoreboard['adjudication_status'].eq('nuclear_localized')
].copy()
best_primary_model = (
    str(nuclear_localized.sort_values('enrichment_gap', ascending=False)['model_label'].iloc[0])
    if not nuclear_localized.empty
    else 'none'
)

if not nuclear_localized.empty:
    claim_status = 'localized_covariance_nuclear'
    claim_reason = (
        'A compact localized covariance rule survives global scalar controls, local scalar controls, chemistry controls, '
        'and the strict structural witness. The rescue is now strong enough for a bounded primary claim.'
    )
elif bool((final_localized_scoreboard['adjudication_status'] == 'supportive_localized').any()):
    claim_status = 'localized_covariance_supportive'
    claim_reason = (
        'Block 12D sharpens the story substantially, but the final primary claim still falls short of a nuclear localized regime.'
    )
else:
    claim_status = 'covariance_adjudication_still_mixed'
    claim_reason = (
        'Even after the final localized search, the notebook does not find a rule that fully clears the nuclear localized firewall.'
    )

summary_payload = {
    'notebook_slug': NOTEBOOK_SLUG,
    'run_at_utc': RUN_AT,
    'account_label': ACCOUNT_LABEL,
    'claim_status': claim_status,
    'claim_reason': claim_reason,
    'block12c_claim_status': block12c_artifact_summary.get('claim_status', 'missing'),
    'n_models_evaluated': int(len(final_localized_scoreboard)),
    'n_nuclear_localized_models': int(final_localized_scoreboard['adjudication_status'].eq('nuclear_localized').sum()),
    'n_supportive_localized_models': int(final_localized_scoreboard['adjudication_status'].eq('supportive_localized').sum()),
    'best_primary_model': best_primary_model,
    'best_primary_panel': (
        str(nuclear_localized.sort_values('enrichment_gap', ascending=False)['panel_name'].iloc[0])
        if not nuclear_localized.empty
        else 'none'
    ),
    'structural_source_kind': structural_source_kind,
    'structural_source_path': structural_source_path,
}

response_md = (
    f"# Block 12D Final Localization Summary\n\n"
    f"- Claim status: **{claim_status}**\n"
    f"- Claim reason: {claim_reason}\n"
    f"- Best primary model: **{best_primary_model}**\n"
    f"- Nuclear localized models: **{int(final_localized_scoreboard['adjudication_status'].eq('nuclear_localized').sum())}**\n"
    f"- Supportive localized models: **{int(final_localized_scoreboard['adjudication_status'].eq('supportive_localized').sum())}**\n"
    f"- Structural source: `{structural_source_kind}` from `{structural_source_path}`\n"
)

claim_paragraph = (
    'Block 12D is the final localization pass. Instead of asking covariance to prove universality everywhere at once, '
    'the notebook searches for a compact localized rescue regime that clears stronger matched controls and a strict structural witness. '
    'The resulting claim is intentionally bounded: the notebook either upgrades one model into a localized nuclear rescue or it records, cleanly and explicitly, that even the last localization attempt stays supportive.'
)

(MANIFESTS_DIR / 'block12d_final_localization_summary.json').write_text(
    json.dumps(summary_payload, indent=2),
    encoding='utf-8',
)
(MANIFESTS_DIR / 'artifact_summary.json').write_text(
    json.dumps(summary_payload, indent=2),
    encoding='utf-8',
)
(TEXT_DIR / 'block12d_final_localization_summary.md').write_text(response_md + '\n', encoding='utf-8')
(TEXT_DIR / 'block12d_claim_paragraph.md').write_text(claim_paragraph + '\n', encoding='utf-8')

bundle_sources = [TABLES_DIR, FIGURES_DIR, TEXT_DIR, MANIFESTS_DIR, RUNTIME_DIR, LIVE_SCORES_DIR]
if BUNDLE_ROOT.exists():
    shutil.rmtree(BUNDLE_ROOT)
BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
for folder in bundle_sources:
    shutil.copytree(folder, BUNDLE_ROOT / folder.name, dirs_exist_ok=True)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in bundle_sources:
        for file_path in folder.rglob('*'):
            if file_path.is_file():
                archive.write(file_path, arcname=str(file_path.relative_to(RESULTS_ROOT)))

print(json.dumps(summary_payload, indent=2))
display(final_localized_scoreboard)
display(nuclear_localized if not nuclear_localized.empty else pd.DataFrame([{'status': 'no_nuclear_localized_rule_found'}]))
done('Figures, manifests, markdown summaries, and the final Block 12D bundle are written.')

print("TERMINEI PODE SEGUIR")


{
  "notebook_slug": "12d_block12_final_nuclear_localization_h100",
  "run_at_utc": "2026-04-20T16:07:49.092799+00:00",
  "account_label": "local_run",
  "claim_status": "localized_covariance_supportive",
  "claim_reason": "Block 12D sharpens the story substantially, but the final primary claim still falls short of a nuclear localized regime.",
  "block12c_claim_status": "covariance_adjudication_mixed",
  "n_models_evaluated": 10,
  "n_nuclear_localized_models": 0,
  "n_supportive_localized_models": 1,
  "best_primary_model": "none",
  "best_primary_panel": "none",
  "structural_source_kind": "strict",
  "structural_source_path": "C:\\Users\\Davib\\OneDrive\\\u00c1rea de Trabalho\\Stanford-Claw4s\\New Notebooks\\results\\07b_block10_structural_dissociation_tp53_h100\\07b_block10_structural_dissociation_tp53_h100\\tables\\tp53_structural_pairs_variant_level_strict.csv"
}


,model_label,family_label,architecture_kind,panel_name,scope_label,panel_size,coverage_target,coverage_pass,rule_family,gate_name,...,structural_ll_spearman,structural_geometry_gap,block12c_status,adjudication_status,adjudication_reason,status_rank,same_position_rank,positive_witness_panels_present,positive_witness_hit_fraction_mean,primary_claim_firewall
0,ESM-1v,esm_variant_specialist,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,-0.2,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.000000,supportive_or_excluded
1,ESM2-150M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,exclude_chemistry,...,0.4,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.000000,supportive_or_excluded
2,ESM2-35M,esm,masked_encoder,full_tp53,global,255,13,False,pair_only,none,...,0.0,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,0,3,0.166667,supportive_or_excluded
3,ESM2-3B,esm,masked_encoder,dna_binding_core,localized,145,12,True,pair_rank_fixed,none,...,0.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.833333,supportive_or_excluded
4,ESM2-650M,esm,masked_encoder,full_tp53,global,255,13,True,pair_only,none,...,0.4,0.333333,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,3,0.166667,supportive_or_excluded
5,ProGen2-base,progen,causal_decoder,dna_binding_core,localized,145,12,True,pair_only,none,...,1.0,NaN,supportive,supportive_residual,positive_gap_without_full_control_dominance,2,0,0,NaN,supportive_or_excluded
6,ProGen2-small,progen,causal_decoder,full_tp53,global,255,13,True,pair_only,none,...,-0.4,0.333333,supportive,supportive_localized,localized_rule_positive_but_not_fully_nuclear,1,1,0,NaN,supportive_or_excluded
7,ProtBERT,bert_style,bidirectional_encoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.4,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3,0,NaN,supportive_or_excluded
8,ProtBERT-BFD,bert_style,bidirectional_encoder,dna_binding_core,localized,145,12,False,pair_rank_fixed,none,...,0.8,NaN,not_eligible,not_eligible,coverage_or_control_failure,3,1,0,NaN,supportive_or_excluded
9,ProtT5,prottrans,encoder_decoder,full_tp53,global,255,13,True,pair_rank_fixed,none,...,-0.8,NaN,not_eligible,supportive_residual,positive_gap_without_full_control_dominance,2,3,0,NaN,supportive_or_excluded


,status
0,no_nuclear_localized_rule_found


Figures, manifests, markdown summaries, and the final Block 12D bundle are written.
TERMINEI PODE SEGUIR
TERMINEI PODE SEGUIR
